# Fine-Tuning Whisper-Small on Egyptian Arabic (4GB VRAM Optimized)

This notebook implements Knowledge Distillation:
1. Teacher: Whisper-Large-v3
2. Student: Whisper-Small

Optimization: 4-bit QLoRA with JIT Audio Loading.

In [1]:
import os
import sys
import torch
import librosa
import numpy as np
from datasets import DatasetDict
from transformers import (
    WhisperProcessor, 
    WhisperForConditionalGeneration, 
    Seq2SeqTrainingArguments, 
    Seq2SeqTrainer,
    BitsAndBytesConfig
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# Add parent directory to path to import our custom modules
sys.path.append(os.path.abspath('..'))
from data_loader import EgyptDatasetBuilder

d:\study\4th year 2nd term\NLP\Project 2 - Audio\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Dataset Loading (Path Only)

In [2]:
MODEL_NAME = "openai/whisper-small"
AUDIO_DIR = r"d:\study\4th year 2nd term\NLP\Project 2 - Audio\cut_clips9"
TRANS_DIR = r"d:\study\4th year 2nd term\NLP\Project 2 - Audio\cut_clips9_transcriptions"

processor = WhisperProcessor.from_pretrained(MODEL_NAME, language="arabic", task="transcribe")

builder = EgyptDatasetBuilder(AUDIO_DIR, TRANS_DIR)
dataset = builder.build()
dataset = dataset.train_test_split(test_size=100, seed=42)

print(f"Dataset ready. Train samples: {len(dataset['train'])}")

📦 Found 7502 audio files. Matching with transcriptions...
✅ Successfully matched 7501 pairs.
Dataset ready. Train samples: 7401


## 2. Model Initialization (4-bit QLoRA)

In [3]:
print(f"Loading {MODEL_NAME} in 4-bit...")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

model = WhisperForConditionalGeneration.from_pretrained(
    MODEL_NAME, 
    quantization_config=bnb_config, 
    device_map='auto'
)

model = prepare_model_for_kbit_training(model)

config = LoraConfig(
    r=32, 
    lora_alpha=64, 
    target_modules=['q_proj', 'v_proj'], 
    lora_dropout=0.05, 
    bias='none'
)

# Force the model to always transcribe in Arabic
model.config.forced_decoder_ids = None
model.config.suppress_tokens = []
model.generation_config.language = "arabic"
model.generation_config.task = "transcribe"

# This stops the 'forced_decoder_ids' deprecation warning
model.generation_config.forced_decoder_ids = None


model = get_peft_model(model, config)
model.print_trainable_parameters()

Loading openai/whisper-small in 4-bit...


Loading weights: 100%|██████████| 479/479 [00:04<00:00, 118.78it/s]


trainable params: 3,538,944 || all params: 245,273,856 || trainable%: 1.4429


## 3. Training Configuration (JIT Data Collator)

In [3]:
## 3. Training Configuration (JIT Data Collator & Metrics)
from dataclasses import dataclass
from typing import Any, Dict, List, Union
import librosa
import evaluate

# A. Metrics Helper (Calculates Word Error Rate)
metric = evaluate.load("wer")

def compute_metrics(pred):
    pred_ids = pred.predictions
    label_ids = pred.label_ids

    # Replace -100 with pad_token_id
    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id

    # Decode predictions and labels
    pred_str = processor.batch_decode(pred_ids, skip_special_tokens=True)
    label_str = processor.batch_decode(label_ids, skip_special_tokens=True)

    # Calculate WER (scaled by 100 for percentage)
    wer = 100 * metric.compute(predictions=pred_str, references=label_str)
    return {"wer": wer}

# B. Data Collator (Just-In-Time processing)
@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any
    def __call__(self, features: List[Dict[str, Any]]) -> Dict[str, torch.Tensor]:
        input_features = []
        label_features = []
        for feature in features:
            audio_path = feature['audio']
            audio_array, _ = librosa.load(audio_path, sr=16000)
            f = self.processor.feature_extractor(audio_array, sampling_rate=16000).input_features[0]
            input_features.append({'input_features': f})
            l = self.processor.tokenizer(feature['sentence']).input_ids
            label_features.append({'input_ids': l})

        batch = self.processor.feature_extractor.pad(input_features, return_tensors='pt')
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors='pt')
        labels = labels_batch['input_ids'].masked_fill(labels_batch.attention_mask.ne(1), -100)
        if (labels[:, 0] == self.processor.tokenizer.bos_token_id).all().cpu().item():
            labels = labels[:, 1:]
        batch['labels'] = labels
        return batch

data_collator = DataCollatorSpeechSeq2SeqWithPadding(processor=processor)

# C. Training Arguments (Configured for 1 Epoch)
training_args = Seq2SeqTrainingArguments(
    output_dir='./whisper-small-egyptian-lora',
    num_train_epochs=1,               # Set to 1 Epoch
    max_steps=-1,                     # Required to use epochs instead of steps
    per_device_train_batch_size=1,
    gradient_accumulation_steps=16,
    learning_rate=1e-4,
    warmup_steps=50,
    gradient_checkpointing=True,
    fp16=True,
    eval_strategy='steps',
    per_device_eval_batch_size=1,
    predict_with_generate=True,
    generation_max_length=180,
    save_steps=100,                    # Multiple of eval_steps
    eval_steps=25,                    
    logging_steps=10,
    report_to=['tensorboard'],
    load_best_model_at_end=True,
    metric_for_best_model='wer',
    greater_is_better=False,
    push_to_hub=False,
    remove_unused_columns=False,
)

# D. Initialize Trainer
trainer = Seq2SeqTrainer(
    args=training_args,
    model=model,
    train_dataset=dataset['train'],
    eval_dataset=dataset['test'],
    data_collator=data_collator,
    processing_class=processor.feature_extractor,
    compute_metrics=compute_metrics,    # Passes our new metric function
)


NameError: name 'processor' is not defined

## 4. Training and Visualization

In [5]:
%load_ext tensorboard
%tensorboard --logdir ./whisper-small-egyptian-lora/runs

print("Starting Training...")
trainer.train()

Reusing TensorBoard on port 6006 (pid 14524), started 5:10:20 ago. (Use '!kill 14524' to kill it.)

Starting Training...


Step,Training Loss,Validation Loss,Wer
25,80.190302,3.926848,72.764228
50,34.561191,1.774389,58.943089
75,24.899928,1.399301,59.552846
100,20.002255,1.247956,53.455285
125,17.649811,1.032349,52.845528
150,11.290193,0.648521,53.252033
175,7.972687,0.497352,52.032520
200,7.451966,0.488377,50.203252
225,7.062353,0.462892,48.577236
250,6.452200,0.463226,47.560976


[transformers] The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
[transformers] A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> will take precedence. Please check the docstring of <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> to see related `.generate()` flags.
[transformers] A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transform

TrainOutput(global_step=463, training_loss=15.692121645801784, metrics={'train_runtime': 61310.8885, 'train_samples_per_second': 0.121, 'train_steps_per_second': 0.008, 'total_flos': 2.17353663111168e+18, 'train_loss': 15.692121645801784, 'epoch': 1.0})

## 5. Comparison Suite (Base vs Fine-tuned vs Large)

In [8]:
import sounddevice as sd
from scipy.io.wavfile import write
from IPython.display import Audio, display, HTML
from transformers import pipeline

def record_audio(filename='test_input.wav', duration=5):
    print(f"Recording for {duration} seconds...")
    fs = 16000
    recording = sd.rec(int(duration * fs), samplerate=fs, channels=1, dtype='float32')
    sd.wait()
    write(filename, fs, recording)
    print(f"Saved to {filename}")
    display(Audio(filename))

def run_comparison(audio_path='test_input.wav'):
    audio, _ = librosa.load(audio_path, sr=16000)
    results = {}
    
    # 1. Fine-tuned Student (in memory)
    input_features = processor(audio, sampling_rate=16000, return_tensors='pt').input_features.to('cuda')
    with torch.no_grad():
        generated_ids = model.generate(input_features)
    results['Fine-tuned Student'] = processor.batch_decode(generated_ids, skip_special_tokens=True)[0]

    # 2. Original Base Small
    print("Loading Original Small...")
    base_model = WhisperForConditionalGeneration.from_pretrained(
        'openai/whisper-small', 
        quantization_config=bnb_config, 
        device_map='auto'
    )
    with torch.no_grad():
        base_ids = base_model.generate(input_features)
    results['Original Small'] = processor.batch_decode(base_ids, skip_special_tokens=True)[0]
    del base_model
    torch.cuda.empty_cache()

    # 3. Teacher Large-v3
    print("Loading Teacher Large-v3...")
    teacher_pipe = pipeline(
        'automatic-speech-recognition', 
        model='openai/whisper-large-v3', 
        model_kwargs={'quantization_config': bnb_config}, 
        device_map='auto'
    )
    teacher_out = teacher_pipe(audio)
    results['Teacher Large-v3'] = teacher_out['text']
    del teacher_pipe
    torch.cuda.empty_cache()

    # Display Results
    html = f'''
    <div style="padding:15px; border:1px solid #ddd; border-radius:10px;">
        <h3>Comparison Results</h3>
        <p><b>Original Small:</b> {results['Original Small']}</p>
        <p style="color:green;"><b>Fine-tuned Student:</b> {results['Fine-tuned Student']}</p>
        <p style="color:blue;"><b>Teacher Large-v3:</b> {results['Teacher Large-v3']}</p>
    </div>
    '''
    display(HTML(html))

# Usage:
# record_audio(duration=5)
# run_comparison('test_input.wav')

In [ ]:
run_comparison('test_input.wav')

Loading Original Small...


Loading weights: 100%|██████████| 479/479 [00:03<00:00, 137.24it/s]
[transformers] Using custom `forced_decoder_ids` from the (generation) config. This is deprecated in favor of the `task` and `language` flags/config options.
[transformers] Transcription using a multilingual Whisper will default to language detection followed by transcription instead of translation to English. This might be a breaking change for your use case. If you want to instead always translate your audio to English, make sure to pass `language='en'`. See https://github.com/huggingface/transformers/pull/28687 for more details.


Loading Teacher Large-v3...


Loading weights: 100%|██████████| 1259/1259 [00:21<00:00, 58.99it/s]


In [ ]:
# 1. Save the fine-tuned LoRA adapter
final_model_path = "./whisper-small-egyptian-final"
trainer.save_model(final_model_path)

# 2. Save the processor (important for future inference)
processor.save_pretrained(final_model_path)

print(f"✅ Model and processor saved to {final_model_path}")

✅ Model and processor saved to ./whisper-small-egyptian-final


# Pruning Adapeters

In [1]:
import torch.nn.utils.prune as prune

def apply_pruning_to_lora(model, amount=0.3):
    """
    Prunes a percentage of the LoRA adapter weights.
    amount: float between 0.0 and 1.0 (e.g., 0.3 = 30% pruning)
    """
    print(f"Applying {amount*100}% Magnitude Pruning to LoRA layers...")
    
    # We target the LoRA 'A' and 'B' matrices in the attention layers
    for name, module in model.named_modules():
        if "lora_A" in name or "lora_B" in name:
            if isinstance(module, torch.nn.Linear):
                prune.l1_unstructured(module, name="weight", amount=amount)
                # Make the pruning permanent
                prune.remove(module, "weight")
    
    print("Pruning complete.")
    return model

In [2]:
import time
import torch
import librosa
import pandas as pd
import torch.nn.utils.prune as prune
from IPython.display import display, HTML
from transformers import logging, WhisperProcessor, WhisperForConditionalGeneration
from peft import PeftModel

# --- CONFIGURATION ---
MODEL_FOLDER = "./whisper-small-egyptian-lora-final"
BASE_MODEL_NAME = "openai/whisper-small"
# Automatically detect device (uses CPU if no GPU is found)
device = "cuda" if torch.cuda.is_available() else "cpu"

# 1. Silence Transformer warnings
logging.set_verbosity_error() 

# 2. Load Model and Processor
print(f"Loading model from {MODEL_FOLDER} onto {device.upper()}...")
processor = WhisperProcessor.from_pretrained(MODEL_FOLDER)

# Note: We load without quantization (no BitsAndBytes) to ensure CPU compatibility
model = WhisperForConditionalGeneration.from_pretrained(
    BASE_MODEL_NAME, 
    torch_dtype=torch.float32, # Use float32 for CPU
    low_cpu_mem_usage=True
)
model = PeftModel.from_pretrained(model, MODEL_FOLDER)
model.to(device)
model.eval()

# Force decoder settings
model.generation_config.max_length = None
model.generation_config.language = "arabic"
model.generation_config.task = "transcribe"

# 3. Prepare 20 samples from the evaluation set
test_subset = dataset["test"].select(range(min(20, len(dataset["test"]))))

# --- Helper Functions ---

def count_nonzero_params(model):
    """Calculates non-zero vs total parameters for sparse size estimation"""
    nonzero = 0
    total = 0
    for param in model.parameters():
        nonzero += torch.count_nonzero(param).item()
        total += param.nelement()
    return nonzero, total

def get_stats():
    """Captures current Memory usage and Theoretical Sparse Size"""
    # VRAM for GPU, 0 for CPU
    mem_usage = torch.cuda.memory_allocated() / 1024**2 if torch.cuda.is_available() else 0
    nonzero, _ = count_nonzero_params(model)
    theoretical_mb = (nonzero * 4) / 1024**2 # 4 bytes per float32
    return mem_usage, theoretical_mb

def get_lora_state_dict(model):
    """Safely extracts only LoRA adapter weights"""
    return {k: v.cpu().clone() for k, v in model.named_parameters() if "lora" in k}

def run_mini_benchmark(active_model, desc):
    """Runs inference on the subset and returns averaged metrics"""
    predictions = []
    references = []
    
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        
    start_mem, theoretical_size = get_stats()
    start_time = time.time()
    
    for item in test_subset:
        audio_path = item["audio"]
        audio, _ = librosa.load(audio_path, sr=16000)
        input_features = processor(audio, sampling_rate=16000, return_tensors="pt").input_features.to(device)
        
        with torch.no_grad():
            generated_ids = active_model.generate(
                input_features, 
                max_new_tokens=128
            )
        
        pred = processor.batch_decode(generated_ids, skip_special_tokens=True)[0]
        predictions.append(pred)
        references.append(item["sentence"])
    
    elapsed = time.time() - start_time
    # Uses 'metric' (WER) defined in your notebook
    wer = 100 * metric.compute(predictions=predictions, references=references)
    
    return {
        "Configuration": desc,
        "WER (%)": f"{wer:.2f}%",
        "Total Time (s)": f"{elapsed:.2f}s",
        "Avg Latency (s)": f"{elapsed/len(test_subset):.2f}s",
        "Memory (MB)": f"{start_mem:.0f} MB" if torch.cuda.is_available() else "N/A (CPU)",
        "Sparse Size (MB)": f"{theoretical_size:.1f} MB"
    }

# --- MASTER EXECUTION LOOP ---
results = []

# Save a clean copy of the LoRA weights to restore after pruning
original_lora_state = get_lora_state_dict(model)

def restore_lora():
    model.load_state_dict(original_lora_state, strict=False)

# TEST 1: Baseline (No Pruning)
print("Testing Baseline...")
results.append(run_mini_benchmark(model, "Baseline (Full Fine-tuned)"))

# TEST 2: Magnitude Pruning (30%)
# Assuming apply_pruning_to_lora is defined in a previous cell
apply_pruning_to_lora(model, amount=0.3)
results.append(run_mini_benchmark(model, "Magnitude Pruned (30%)"))
restore_lora()

# TEST 3: Magnitude Pruning (60%)
apply_pruning_to_lora(model, amount=0.6)
results.append(run_mini_benchmark(model, "Magnitude Pruned (60%)"))
restore_lora()

# TEST 4: Layer Pruning (Drop 2 Decoder Layers)
print("Testing Layer Pruning...")
# Whisper decoder structure path:
original_decoder_layers = model.base_model.model.model.decoder.layers
model.base_model.model.model.decoder.layers = torch.nn.ModuleList(list(original_decoder_layers)[:-2])
results.append(run_mini_benchmark(model, "Layer Pruned (-2 Decoder Layers)"))
model.base_model.model.model.decoder.layers = original_decoder_layers # Restore

# 📊 FINAL COMPARISON TABLE
df_master = pd.DataFrame(results)
display(HTML("<h2 style='color: #2c3e50;'>🚀 Master Efficiency Benchmark Results</h2>"))
display(df_master)


d:\study\4th year 2nd term\NLP\Project 2 - Audio\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


NameError: name 'model' is not defined

# Comprehensive Comparison

In [37]:
record_audio(duration=5)

Recording for 5 seconds...
Saved to test_input.wav


In [41]:
import time
import os
import torch
import librosa
import evaluate
import pandas as pd
from IPython.display import display, HTML
from transformers import pipeline, WhisperForConditionalGeneration

# 1. 📋 CONFIGURE TEST DATA
COMPARISON_FOLDER = "audio_comparison"
TRUE_LABELS = {
    "sample1.wav": "منور يا باشا عامل ايه",
    "sample2.wav": "دلوقتي هنبدا الدرس التاني",
    "sample3.wav": "يا ريت تفكك من الحوارات دي علشان مش جايبة همها",
    "sample4.wav": "هو كل شوية يطنشني ويقولي هبقى أكلمك بعدين",
    "sample5.wav": "إيه يا عم القفش ده؟ خليك فريش شوية",
    "sample6.wav": "الموضوع بييجي خبط لزق كده من غير مقدمات",
    "sample7.wav": "على وضعك يا يعم ومتشيلش من حد",
    "sample8.wav": "بقولك إيه سيبك من اللت والعجن الكتير ده",
    "sample9.wav": "عاش يا وحش التطور ده بييجي بالممارسة",
    "sample10.wav": "متكبرهاش في دماغك بقى وصفي نيتك",
}

wer_metric = evaluate.load("wer")

def get_model_size_mb(model):
    param_size = sum(p.nelement() * p.element_size() for p in model.parameters())
    return param_size / 1024**2

def benchmark_models():
    models_to_test = [
        {"name": "Original Small", "path": "openai/whisper-small", "type": "base"},
        {"name": "Fine-tuned Student", "type": "lora"}, 
        {"name": "Pruned Student (30%)", "type": "lora_pruned", "amount": 0.3},
        {"name": "Teacher Large-v3", "path": "openai/whisper-large-v3", "type": "pipe"}
    ]
    
    overall_results = []
    # This will store: { "sample1.wav": {"Reference": "...", "Original Small": "...", ...} }
    transcription_comparison = {k: {"Reference": v} for k, v in TRUE_LABELS.items()}
    
    original_lora_state = {k: v.cpu().clone() for k, v in model.named_parameters() if "lora" in k}

    for m_cfg in models_to_test:
        print(f"🚀 Benchmarking: {m_cfg['name']}...")
        current_model, pipe = None, None
        
        # --- LOAD ---
        if m_cfg['type'] == "base":
            current_model = WhisperForConditionalGeneration.from_pretrained(
                m_cfg['path'], quantization_config=bnb_config, device_map="auto"
            )
        elif m_cfg['type'] == "lora":
            current_model = model
        elif m_cfg['type'] == "lora_pruned":
            current_model = model
            apply_pruning_to_lora(current_model, amount=m_cfg['amount'])
        elif m_cfg['type'] == "pipe":
            pipe = pipeline("automatic-speech-recognition", model=m_cfg['path'], 
                            model_kwargs={"quantization_config": bnb_config}, device_map="auto")

        # --- STATS ---
        torch.cuda.empty_cache()
        vram = torch.cuda.memory_allocated() / 1024**2
        model_size = get_model_size_mb(current_model) if current_model else 1550

        # --- INFERENCE ---
        preds, refs = [], []
        start_time = time.time()

        for filename, truth in TRUE_LABELS.items():
            path = os.path.join(COMPARISON_FOLDER, filename)
            if not os.path.exists(path): continue
            
            audio, _ = librosa.load(path, sr=16000)
            if pipe:
                pred = pipe(audio)["text"]
            else:
                input_features = processor(audio, sampling_rate=16000, return_tensors="pt").input_features.to("cuda")
                with torch.no_grad():
                    gen_ids = current_model.generate(input_features, max_new_tokens=128)
                pred = processor.batch_decode(gen_ids, skip_special_tokens=True)[0]
            
            preds.append(pred)
            refs.append(truth)
            # Store for the transcription table
            transcription_comparison[filename][m_cfg['name']] = pred

        # --- METRICS ---
        if preds:
            elapsed = time.time() - start_time
            wer = 100 * wer_metric.compute(predictions=preds, references=refs)
            overall_results.append({
                "Model": m_cfg['name'], "WER (%)": f"{wer:.2f}%",
                "Avg Time (s)": f"{elapsed/len(preds):.2f}s",
                "VRAM (MB)": f"{vram:.0f} MB", "Size (MB)": f"{model_size:.0f} MB"
            })
        
        # --- CLEANUP ---
        if m_cfg['type'] == "lora_pruned":
            model.load_state_dict(original_lora_state, strict=False)
        if m_cfg['type'] in ["base", "pipe"]:
            del current_model, pipe
            torch.cuda.empty_cache()

    # --- RENDER STYLING ---
    styles = """
    <style>
        .benchmark-container { background-color: #1e1e2e; color: #cdd6f4; padding: 20px; border-radius: 15px; font-family: 'Segoe UI', sans-serif; }
        .premium-table { border-collapse: collapse; width: 100%; background-color: #313244; border-radius: 10px; overflow: hidden; margin-bottom: 20px; }
        .premium-table th { background-color: #45475a; color: #89b4fa; padding: 15px; text-align: left; text-transform: uppercase; font-size: 0.85em; }
        .premium-table td { border-bottom: 1px solid #45475a; padding: 12px; color: #cdd6f4; }
        .premium-table tr:hover { background-color: #45475a; transition: 0.3s; }
        .ref-col { color: #a6e3a1 !important; font-weight: bold; }
        h2 { color: #f5c2e7; margin-bottom: 10px; }
    </style>
    """

    # 📊 1. METRICS SUMMARY
    df_metrics = pd.DataFrame(overall_results)
    
    # 📝 2. TRANSCRIPTION DETAILS
    # Flatten the nested dict for pandas
    detailed_data = []
    for fname, data in transcription_comparison.items():
        row = {"File": fname}
        row.update(data)
        detailed_data.append(row)
    df_details = pd.DataFrame(detailed_data)

    display(HTML(styles + f"""
    <div class='benchmark-container'>
        <h2>🚀 Model Performance Metrics</h2>
        {df_metrics.to_html(index=False, classes='premium-table')}
        
        <h2>📝 Detailed Transcriptions Comparison</h2>
        {df_details.to_html(index=False, classes='premium-table')}
    </div>
    """))

In [42]:
benchmark_models()

🚀 Benchmarking: Original Small...


Loading weights: 100%|██████████| 479/479 [00:04<00:00, 100.15it/s]


🚀 Benchmarking: Fine-tuned Student...
🚀 Benchmarking: Pruned Student (30%)...
Applying 30.0% Magnitude Pruning to LoRA layers...
Pruning complete.
🚀 Benchmarking: Teacher Large-v3...


Loading weights: 100%|██████████| 1259/1259 [00:17<00:00, 73.14it/s]
